# Immune-signature enrichment (AUCell) — cord blood atlas
Reads `Seurat.rds` → builds immune gene sets (MSigDB) → **restricts them to the panel genes** →
AUCell per cell → keeps signatures that vary across cell types (Kruskal-Wallis + BH) → averages
per cell type → z-scored **pheatmap**.

In [1]:
# install.packages(c("msigdbr","viridis","RColorBrewer","scales"))
# if (!requireNamespace("BiocManager", quietly=TRUE)) install.packages("BiocManager")
# BiocManager::install("AUCell")

In [2]:
suppressMessages({
  library(Seurat); library(AUCell); library(msigdbr)
  library(pheatmap); library(viridis); library(RColorBrewer); library(scales)
})
set.seed(1)

# ---------------- params ----------------
RDS       <- "Seurat_CellChat.rds"     # full atlas (RNA/SCT/ADT + CellType + Sample)
MIN_GENES <- 5               # keep a signature only if >= this many of its genes are on the panel
PADJ      <- 0.01            # Kruskal-Wallis BH cutoff to call a signature "informative"
TOP_N     <- 100              # after selection, keep only the TOP_N most cell-type-variable signatures
OUT_CSV   <- "immune_signatures_AUC.csv"
OUT_JPEG  <- "immune_signature_heatmap.jpeg"

In [3]:
obj    <- readRDS(RDS)
panel  <- rownames(GetAssayData(obj, assay = "RNA", layer = "counts"))   # measured genes = the panel universe
counts <- as.matrix(GetAssayData(obj, assay = "RNA", layer = "counts"))  # genes x cells
ct     <- as.character(obj$CellType)
cell_order <- if (is.factor(obj$CellType)) levels(obj$CellType) else sort(unique(ct))
cat(sprintf("cells %d | panel genes %d | cell types %d\n", ncol(counts), length(panel), length(cell_order)))

cells 22046 | panel genes 389 | cell types 14


In [4]:
# ---------------- gene sets (MSigDB) ----------------
# robust to msigdbr API change (collection/subcollection vs category/subcategory)
msig <- function(cat, sub = NULL) tryCatch(
  msigdbr(species = "Homo sapiens", collection = cat, subcollection = sub),
  error = function(e) msigdbr(species = "Homo sapiens", category = cat, subcategory = sub))

# choose collections here. GO:BP alone keeps it focused; add c("H") or c("C2","CP") to widen.
COLLECTIONS <- list(c("C5", "GO:BP"))          # e.g. list(c("H"), c("C5","GO:BP"))
df <- do.call(rbind, lapply(COLLECTIONS, function(x) msig(x[1], if (length(x) > 1) x[2] else NULL)))

# USE_ALL = TRUE -> keep every set (panel coverage + Kruskal-Wallis auto-detect the relevant ones)
# USE_ALL = FALSE -> pre-filter by immune + erythroid keywords (faster, narrower)
USE_ALL <- TRUE
kw <- paste0("IMMUN|INFLAMMAT|CYTOKINE|CHEMOKINE|INTERFERON|INTERLEUKIN|COMPLEMENT|LEUKOCYTE|",
             "LYMPHOCYTE|T_CELL|B_CELL|NK_|NATURAL_KILLER|ANTIGEN|MHC|TLR|TOLL|TCR|BCR|INNATE|",
             "ADAPTIVE|CD8|CD4|TREG|",
             "ERYTHRO|HEME|HEMOGLOBIN|PORPHYRIN|GATA1|HEMATOPOIE|OXYGEN_TRANSPORT|IRON")
if (!USE_ALL) df <- subset(df, grepl(kw, gs_name, ignore.case = TRUE))

sym <- if ("gene_symbol" %in% names(df)) df$gene_symbol else df[[grep("symbol", names(df), ignore.case=TRUE, value=TRUE)[1]]]
gs  <- split(sym, df$gs_name)

# restrict each set to panel genes, drop sparsely-covered sets
gs <- lapply(gs, function(g) intersect(unique(g), panel))
gs <- gs[vapply(gs, length, 0L) >= MIN_GENES]
cat(sprintf("candidate sets: %d total -> %d with >= %d panel genes\n",
            length(unique(df$gs_name)), length(gs), MIN_GENES))

candidate sets: 7538 total -> 1374 with >= 5 panel genes


In [5]:
# ---------------- AUCell ----------------
ranks <- AUCell_buildRankings(counts, plotStats = FALSE, verbose = FALSE)
auc   <- AUCell_calcAUC(gs, ranks, aucMaxRank = ceiling(1 * nrow(ranks)))  # all genes
aucm  <- getAUC(auc)                                     # signatures x cells
stopifnot(all(colnames(aucm) == colnames(counts)))
cat(sprintf("AUC matrix: %d signatures x %d cells\n", nrow(aucm), ncol(aucm)))

AUC matrix: 1374 signatures x 22046 cells


In [6]:
# ---------------- select signatures that differ across cell types ----------------
grp  <- factor(ct, levels = cell_order)
kwp  <- apply(aucm, 1, function(x) tryCatch(kruskal.test(x ~ grp)$p.value, error = function(e) NA))
padj <- p.adjust(kwp, "BH")
sig  <- names(padj)[!is.na(padj) & padj < PADJ]
cat(sprintf("informative signatures (BH p < %.2g): %d / %d\n", PADJ, length(sig), nrow(aucm)))

# rank the survivors by how much they vary across CellType (spread of mean AUC), keep TOP_N
ct_means <- t(apply(aucm[sig, , drop = FALSE], 1, function(x) tapply(x, grp, mean)))
spread   <- apply(ct_means, 1, function(m) max(m) - min(m))
sig      <- names(sort(spread, decreasing = TRUE))[seq_len(min(TOP_N, length(sig)))]
cat(sprintf("kept top %d most cell-type-variable signatures\n", length(sig)))

# ---- mean AUC per CellType x Sample ----
sm  <- as.character(obj$Sample)
smp_order <- if (is.factor(obj$Sample)) levels(obj$Sample) else sort(unique(sm))
cs  <- paste(ct, sm, sep = "_")
cs_levels <- as.vector(t(outer(cell_order, smp_order, paste, sep = "_")))
cs_levels <- cs_levels[cs_levels %in% unique(cs)]
csf <- factor(cs, levels = cs_levels)

avg <- t(apply(aucm[sig, , drop = FALSE], 1, function(x) tapply(x, csf, mean)))
avg <- avg[, cs_levels, drop = FALSE]
write.csv(data.frame(signature = rownames(avg), avg, padj = padj[rownames(avg)], check.names = FALSE),
          OUT_CSV, row.names = FALSE)
dim(avg)

informative signatures (BH p < 0.01): 1374 / 1374
kept top 100 most cell-type-variable signatures


[1] 100  81

In [16]:
# ---------------- z-scored heatmap (CellType x Sample) ----------------
z <- t(scale(t(avg))); z[is.na(z)] <- 0                     # z per signature
z[z >  2] <-  2; z[z < -2] <- -2                            # cap symmetrically for bwr
rn <- gsub("_", " ", sub("^(HALLMARK|REACTOME|GOBP)_", "", rownames(z)))

# column annotation: CellType + Sample (parsed back from CellType_Sample)
ann_ct <- factor(sub("_[^_]*$", "", colnames(z)), levels = cell_order)
ann_sm <- factor(sub("^.*_", "",   colnames(z)), levels = smp_order)
ann_col <- data.frame(CellType = ann_ct, Sample = ann_sm)
rownames(ann_col) <- colnames(z)
ct_pal <- setNames(scales::hue_pal()(length(cell_order)), cell_order)
sm_pal <- setNames(RColorBrewer::brewer.pal(max(3, length(smp_order)), "Dark2")[seq_along(smp_order)], smp_order)
ann_colors <- list(CellType = ct_pal, Sample = sm_pal)

bwr    <- colorRampPalette(c("blue", "white", "red"))(256)  # diverging, white = 0
breaks <- seq(-2, 2, length.out = 257)

pheatmap(
  z,
  color            = bwr,
  breaks           = breaks,
  cluster_rows     = TRUE, cluster_cols = FALSE,
  labels_row       = rn,
  show_colnames    = TRUE,
  annotation_col   = ann_col,
  annotation_colors= ann_colors,
  gaps_col         = head(cumsum(table(ann_ct)[cell_order]), -1),  # split blocks by CellType
  border_color     = NA,
  fontsize_row     = 7, fontsize_col = 7,
  cellwidth        = 10,
  main             = "Immune-signature enrichment (AUCell) by cell type x donor",
  filename         = OUT_JPEG, width = 20, height = 16.5
)
cat("saved", OUT_JPEG, "and", OUT_CSV, "\n")

saved immune_signature_heatmap.jpeg and immune_signatures_AUC.csv 


**Outputs:** `immune_signature_heatmap.jpeg` (signatures × cell types, z-scored) and
`immune_signatures_AUC.csv` (mean AUC per cell type + BH p).

**Knobs:** `MIN_GENES` (panel coverage per set), `PADJ` (KW cutoff), `aucMaxRank` (top-rank window).
For a per-donor version, replace `grp` with `CellType×Sample` and average over that (like your ADT heatmap).
To browse these signatures in Vitessce: `obj[["SIG"]] <- CreateAssayObject(counts = aucm)` and add a
`sig.mtx`/`sig_features.txt` block to the Vitessce export.